# molsim — 分子对接 & 结构分析演示

本 notebook 演示 molsim 的核心功能：
1. **分子读取** — PDB / MOL2 / SDF
2. **能量分析** — 力场能量分解
3. **分子对接** — Monte Carlo + 局部优化
4. **结构分析** — RMSD、RMSF、Ramachandran、二级结构、氢键


In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath('.'))))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# 导入 molsim
import molsim as ms
print(f'molsim version: {ms.__version__}')

## 1. 生成测试分子（无需外部 PDB 文件）

用丙氨酸二肽 (Ace-Ala-Nme) 作为蛋白质骨架演示，苯甲酸作为配体。

In [ ]:
from molsim.core import Atom, Bond, Molecule

# ── 构建一个简单的蛋白质片段（丙氨酸残基）用于演示 ──
def make_alanine_dipeptide():
    """Ace-Ala-Nme backbone from idealized geometry."""
    atoms = [
        # Ace cap
        Atom(0,  'CH3', 'C', 'ACE', 1, 'A', np.array([-2.0, 0.0, 0.0])),
        Atom(1,  'C',   'C', 'ACE', 1, 'A', np.array([-0.6, 0.0, 0.0])),
        Atom(2,  'O',   'O', 'ACE', 1, 'A', np.array([-0.1, 1.1, 0.0])),
        # Ala residue
        Atom(3,  'N',   'N', 'ALA', 2, 'A', np.array([0.3, -1.0, 0.0])),
        Atom(4,  'H',   'H', 'ALA', 2, 'A', np.array([0.0, -2.0, 0.0])),
        Atom(5,  'CA',  'C', 'ALA', 2, 'A', np.array([1.7, -0.8, 0.0])),
        Atom(6,  'CB',  'C', 'ALA', 2, 'A', np.array([2.3, -0.8, -1.4])),
        Atom(7,  'C',   'C', 'ALA', 2, 'A', np.array([2.4,  0.4,  0.6])),
        Atom(8,  'O',   'O', 'ALA', 2, 'A', np.array([1.9,  1.5,  0.6])),
        # Nme cap
        Atom(9,  'N',   'N', 'NME', 3, 'A', np.array([3.7,  0.3,  1.0])),
        Atom(10, 'H',   'H', 'NME', 3, 'A', np.array([4.2,  1.1,  0.7])),
        Atom(11, 'CH3', 'C', 'NME', 3, 'A', np.array([4.4, -0.9,  1.3])),
    ]
    bonds = [
        Bond(0, 1), Bond(1, 2), Bond(1, 3),
        Bond(3, 4), Bond(3, 5), Bond(5, 6),
        Bond(5, 7), Bond(7, 8), Bond(7, 9),
        Bond(9, 10), Bond(9, 11),
    ]
    mol = Molecule('ALA_dipeptide')
    mol.atoms = atoms
    mol.bonds = bonds
    return mol


def make_benzoate():
    """Benzoate ligand (苯甲酸根)."""
    # Aromatic ring + carboxylate, centered at origin
    R = 1.40  # C-C aromatic bond length
    angles = np.linspace(0, 2*np.pi, 7)[:-1]
    atoms = []
    bonds = []
    for i, a in enumerate(angles):
        x, y = R * np.cos(a), R * np.sin(a)
        atoms.append(Atom(i, f'C{i+1}', 'C', 'BNZ', 1, 'L',
                          np.array([x, y, 0.0]), is_hetatm=True))
    # Carboxylate on C1
    atoms.append(Atom(6, 'C7', 'C', 'BNZ', 1, 'L',
                      np.array([R + 1.5, 0.0, 0.0]), is_hetatm=True))
    atoms.append(Atom(7, 'O1', 'O', 'BNZ', 1, 'L',
                      np.array([R + 2.1, 1.1, 0.0]), charge=-0.5, is_hetatm=True))
    atoms.append(Atom(8, 'O2', 'O', 'BNZ', 1, 'L',
                      np.array([R + 2.1, -1.1, 0.0]), charge=-0.5, is_hetatm=True))

    # Ring bonds
    for i in range(6):
        bonds.append(Bond(i, (i+1) % 6, 2 if i % 2 == 0 else 1))
    bonds += [Bond(0, 6), Bond(6, 7, 2), Bond(6, 8)]

    mol = Molecule('benzoate')
    mol.atoms = atoms
    mol.bonds = bonds
    return mol


protein = make_alanine_dipeptide()
ligand  = make_benzoate()

print(f'蛋白片段: {protein}')
print(f'配体: {ligand}')
print(f'\n蛋白质坐标质心: {protein.center_of_mass.round(2)}')
print(f'配体坐标质心: {ligand.center_of_mass.round(2)}')

## 2. 力场参数指定 & 能量计算

In [ ]:
from molsim.forcefield import assign_atom_types, gasteiger_charges
from molsim.forcefield.params import apply_amber_charges
from molsim.energy import interaction_energy

# 指定原子类型 + 电荷
assign_atom_types(protein)
assign_atom_types(ligand)
apply_amber_charges(protein)
gasteiger_charges(ligand)

# 打印原子类型和电荷
print('蛋白质骨架原子:')
for a in protein.atoms[:6]:
    print(f'  {a.name:4s}  type={a.atom_type:4s}  charge={a.charge:+.3f}')

print('\n配体原子:')
for a in ligand.atoms:
    print(f'  {a.name:4s}  element={a.element:2s}  charge={a.charge:+.3f}')

In [ ]:
# 将配体放置在蛋白质附近
ligand.translate(protein.center_of_mass + np.array([3.0, 0.0, 0.0]))

# 计算相互作用能量
e = interaction_energy(protein, ligand)
print('相互作用能量分解:')
for k, v in e.items():
    print(f'  {k:8s}: {v:+8.2f} kcal/mol')

# 可视化
fig, ax = plt.subplots(figsize=(6, 4))
ms.viz.plot_energy_decomposition(
    {'VdW': e['vdw'], 'Electrostatic': e['elec'], 'Total': e['total']},
    title='蛋白-配体相互作用能量',
    ax=ax
)
plt.tight_layout()
plt.show()

## 3. 能量最小化

In [ ]:
from molsim.energy import minimize_energy

lig_copy = ligand.clone()

result = minimize_energy(
    mol=lig_copy,
    receptor=protein,
    max_iter=300,
    verbose=True,
)

print(f'\n最小化结果:')
print(f'  初始能量: {interaction_energy(protein, ligand)["total"]:+.2f} kcal/mol')
print(f'  最终能量: {result["energy"]:+.2f} kcal/mol')
print(f'  迭代次数: {result["n_iter"]}')
print(f'  收敛: {result["success"]}')

## 4. 分子对接

In [ ]:
from molsim.docking import DockingEngine

# 创建对接引擎
engine = DockingEngine(protein, ligand, prepare=True)

# 设置结合位点（以蛋白质质心为中心）
center = protein.center_of_mass
engine.set_binding_site(center=center, radius=8.0)

print(f'结合位点中心: {center.round(2)}')
print(f'搜索半径: 8.0 Å')
print(f'可旋转键数: {len(engine._rot_bonds)}')
print()

# 运行对接
results = engine.dock(
    n_random=500,     # 随机构象数（实际对接建议 2000+）
    n_refine=10,      # 精细优化的前N个构象
    seed=42,
    verbose=True,
)

In [ ]:
# 查看对接结果
print('\n对接结果汇总:')
print(f'{"排名":>4} {"总分":>8} {"VdW":>8} {"静电":>8} {"氢键":>8} {"RMSD":>8}')
print('-' * 50)
for r in results[:8]:
    print(f'{r.rank:4d} {r.score:8.2f} {r.vdw:8.2f} {r.elec:8.2f} '
          f'{r.hbond:8.2f} {r.rmsd_from_top:8.2f}')

# 可视化
fig, ax = plt.subplots(figsize=(9, 4))
ms.viz.plot_docking_results(results, top_n=8, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# 保存最优构象到 PDB
if results:
    best_lig = engine.apply_pose(results[0])
    ms.write_pdb(best_lig, '/tmp/best_pose.pdb', remark=f'Score: {results[0].score:.2f} kcal/mol')
    print('最优对接构象已保存至 /tmp/best_pose.pdb')
    print(f'最优得分: {results[0].score:.2f} kcal/mol')

## 5. 结构分析

In [ ]:
# ── 构建一个模拟的多帧轨迹（模拟结构涨落）──
from molsim.analysis import Trajectory, rmsd, rmsf

traj = Trajectory(protein)
orig_pos = protein.positions.copy()
np.random.seed(0)

for i in range(50):
    noise = np.random.randn(*orig_pos.shape) * 0.3   # 0.3 Å 涨落
    traj.add_frame(orig_pos + noise, energy=-100 + np.random.randn() * 5)

print(f'轨迹: {traj}')

# RMSD 时间序列
rmsd_vals = traj.rmsd_series(ref_frame=0)
print(f'平均 RMSD: {rmsd_vals.mean():.3f} Å')
print(f'最大 RMSD: {rmsd_vals.max():.3f} Å')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3))
ms.viz.plot_rmsd(rmsd_vals, title='骨架 RMSD', ax=ax1)

# RMSF 逐原子
rmsf_vals = traj.rmsf_per_atom()
ms.viz.plot_rmsf(rmsf_vals, title='逐原子 RMSF', ax=ax2)
plt.tight_layout()
plt.show()

In [ ]:
# ── Ramachandran 分析 ──
from molsim.analysis import ramachandran_angles, secondary_structure

# 构建更长的蛋白质片段用于演示（5个残基）
def make_helix_fragment(n_res=8):
    """Generate idealized alpha-helix coordinates."""
    atoms = []
    bonds = []
    idx = 0
    # Ideal alpha helix: rise=1.5 Å/res, twist=100 deg
    rise = 1.5
    twist_deg = 100.0
    r_CA = 2.3  # CA radius from helix axis

    for i in range(n_res):
        z = i * rise
        angle = np.radians(i * twist_deg)
        x_CA = r_CA * np.cos(angle)
        y_CA = r_CA * np.sin(angle)

        N_pos  = np.array([x_CA - 0.7, y_CA - 0.7, z - 0.4])
        CA_pos = np.array([x_CA,       y_CA,       z       ])
        C_pos  = np.array([x_CA + 0.8, y_CA - 0.5, z + 0.5])
        O_pos  = np.array([x_CA + 1.5, y_CA - 1.2, z + 0.3])
        H_pos  = np.array([x_CA - 1.0, y_CA - 1.0, z - 0.7])

        res_id = i + 1
        aa_names = ['ALA', 'GLY', 'VAL', 'LEU', 'ILE', 'SER', 'THR', 'ASP']
        res_name = aa_names[i % len(aa_names)]

        base = idx
        atoms += [
            Atom(idx,   'N',  'N', res_name, res_id, 'A', N_pos),
            Atom(idx+1, 'H',  'H', res_name, res_id, 'A', H_pos),
            Atom(idx+2, 'CA', 'C', res_name, res_id, 'A', CA_pos),
            Atom(idx+3, 'C',  'C', res_name, res_id, 'A', C_pos),
            Atom(idx+4, 'O',  'O', res_name, res_id, 'A', O_pos),
        ]
        bonds += [Bond(base, base+1), Bond(base, base+2),
                  Bond(base+2, base+3), Bond(base+3, base+4)]
        if i > 0:
            bonds.append(Bond(base - 2, base))  # C(i-1) - N(i)
        idx += 5

    mol = Molecule('helix')
    mol.atoms = atoms
    mol.bonds = bonds
    return mol


helix = make_helix_fragment(8)
angles = ramachandran_angles(helix)
ss = secondary_structure(helix)

print('二级结构指定:')
for rid, code in ss.items():
    resname = next(a.residue_name for a in helix.atoms if a.residue_id == rid)
    print(f'  残基 {rid:3d} {resname:3s}: {code} ({["Helix","Sheet","Turn","Coil"][["H","E","T","C"].index(code)]})')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
ms.viz.plot_ramachandran(angles, title='Ramachandran 图', ax=ax1)
ms.viz.plot_secondary_structure(ss, title='二级结构', ax=ax2)
plt.tight_layout()
plt.show()

In [ ]:
# ── 氢键分析 ──
from molsim.analysis import find_hbonds, residue_contacts

# 将最优配体放回蛋白质附近
if results:
    best_lig = engine.apply_pose(results[0])
    hbonds = find_hbonds(engine.receptor, best_lig, d_cutoff=3.5)
    print(f'\n蛋白-配体氢键 (前5个):')
    for hb in hbonds[:5]:
        d = hb['donor_atom']
        a = hb['acceptor_atom']
        print(f'  {d.residue_name}{d.residue_id}:{d.name} -> '
              f'{a.residue_name}{a.residue_id}:{a.name}  '
              f'd(H-A)={hb["distance_H_A"]:.2f} Å  angle={hb["angle_DHA"]:.1f}°')
    if not hbonds:
        print('  (未检测到氢键 — 可能需要更大的蛋白质结构)')

    # 残基接触图
    contacts = residue_contacts(engine.receptor, best_lig, cutoff=6.0)
    print(f'\n蛋白-配体残基接触数: {len(contacts)}')
    for r1, r2, d in contacts[:5]:
        print(f'  残基 {r1} <-> 配体残基 {r2}: {d:.2f} Å')

In [ ]:
# ── RMSD 比较两个结构 ──
from molsim.analysis import aligned_rmsd

# 创建一个轻微扰动的结构
protein2 = protein.clone()
np.random.seed(1)
protein2.positions = protein.positions + np.random.randn(*protein.positions.shape) * 0.5

# 直接 RMSD
raw_rmsd = ms.rmsd(protein.positions, protein2.positions)
print(f'未对齐 RMSD: {raw_rmsd:.3f} Å')

# Kabsch 对齐后 RMSD
aligned_rms, aligned_pos = aligned_rmsd(protein, protein2)
print(f'对齐后 RMSD:  {aligned_rms:.3f} Å')

# 旋转半径
rg = ms.radius_of_gyration(protein)
print(f'回旋半径 Rg:  {rg:.3f} Å')

## 6. 从真实 PDB 文件使用

如果你有真实的 PDB 文件，可以这样使用：

In [ ]:
# 真实 PDB 使用示例（取消注释并替换路径）
"""
import molsim as ms

# 读取蛋白质和配体
receptor = ms.read_pdb('protein.pdb')
ligand   = ms.read_pdb('ligand.pdb')          # 或 ms.read_mol2('ligand.mol2')

# 分子对接
engine = ms.DockingEngine(receptor, ligand)
engine.set_binding_site(center=[x, y, z], radius=12.0)
# 或从已知结合残基定义位点：
# engine.set_binding_site_from_residues([45, 46, 70, 71, 99, 100])

results = engine.dock(n_random=3000, n_refine=25, verbose=True)

# 保存最优构象
best = engine.apply_pose(results[0])
ms.write_pdb(best, 'docked_best.pdb')

# 结构分析
angles = ms.ramachandran_angles(receptor)
ss     = ms.secondary_structure(receptor)
hbonds = ms.find_hbonds(receptor, best)
rg     = ms.radius_of_gyration(receptor)

# 可视化
import matplotlib.pyplot as plt
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
ms.viz.plot_ramachandran(angles, ax=axes[0,0])
ms.viz.plot_secondary_structure(ss, ax=axes[0,1])
ms.viz.plot_docking_results(results, ax=axes[1,0])
ms.viz.plot_energy_decomposition(
    ms.interaction_energy(receptor, best), ax=axes[1,1]
)
plt.tight_layout()
"""

print('提示: 下载真实 PDB 文件用以下命令：')
print('  wget https://files.rcsb.org/download/1HVR.pdb   # HIV蛋白酶')
print('  wget https://files.rcsb.org/download/1AKE.pdb   # 腺苷激酶')

## 总结

| 模块 | 功能 |
|------|------|
| `ms.read_pdb / read_mol2 / read_sdf` | 读取分子结构文件 |
| `ms.interaction_energy(rec, lig)` | VdW + 静电能量分解 |
| `ms.minimize_energy(lig, receptor=rec)` | L-BFGS-B 能量最小化 |
| `DockingEngine.dock(n_random, n_refine)` | Monte Carlo 对接 |
| `ms.ramachandran_angles(protein)` | φ/ψ 二面角 |
| `ms.secondary_structure(protein)` | α螺旋/β折叠识别 |
| `ms.find_hbonds(rec, lig)` | 氢键检测 |
| `ms.aligned_rmsd(mol1, mol2)` | Kabsch 对齐 + RMSD |
| `ms.radius_of_gyration(mol)` | 旋转半径 |
| `Trajectory` | 轨迹 RMSD/RMSF 分析 |